# Sección 2 — Preprocesamiento y Embeddings

In [1]:
import warnings, logging, re, pickle, json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from gensim.models import FastText as GensimFastText
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

for _res, _kind in [("stopwords", "corpora"), ("punkt", "tokenizers")]:
    try:
        nltk.data.find(f"{_kind}/{_res}")
    except LookupError:
        nltk.download(_res, quiet=True)

Path("models").mkdir(exist_ok=True)
Path("artifacts").mkdir(exist_ok=True)

TEXT_COL  = "text"
LABEL_COL = "label"
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX   = 0
UNK_IDX   = 1
RANDOM_SEED = 42

print("INFO | Entorno listo.")

INFO | Entorno listo.


## 2.1 Carga de artefactos del Notebook 1

In [2]:
df = pd.read_parquet("artifacts/raw_df.parquet")

with open("artifacts/eda_config.json") as f:
    eda_cfg = json.load(f)

# El P95 derivado del EDA define el sequence_length; se limita a 400 para
# evitar que un corpus inusualmente largo dispare el uso de memoria de la GPU, pero manteniendo la mayoría de los datos.
SEQUENCE_LENGTH = min(eda_cfg["recommended_seq_len"], 400)
EMBEDDING_DIM   = 200   # Ampliamos el espacio vectorial para capturar el léxico técnico de los CVs

logger.info(
    "Datos cargados — %d muestras, sequence_length=%d, embedding_dim=%d.",
    len(df), SEQUENCE_LENGTH, EMBEDDING_DIM
)
df.head(2)

INFO | Datos cargados — 2483 muestras, sequence_length=400, embedding_dim=200.


,text,label
0,ACCOUNTANT\nSummary\nFinancial Accountant spec...,ACCOUNTANT
1,STAFF ACCOUNTANT\nSummary\nHighly analytical a...,ACCOUNTANT


## Validación de cantidad de CVs por profesión

In [3]:
# Verificamos si hay clases con menos de 3 muestras
class_counts = df["label"].value_counts()
small_classes = class_counts[class_counts < 3]

if not small_classes.empty:
    print("⚠️ ¡Atención! Las siguientes clases tienen muy pocas muestras y romperán la estratificación:")
    print(small_classes)
    print("\nRecomendación: Elimina esas carpetas/muestras o agrega más archivos antes de continuar.")
else:
    print("✅ Todas las clases tienen suficientes muestras. Puedes proceder con el split.")

✅ Todas las clases tienen suficientes muestras. Puedes proceder con el split.


## 2.2 División estratificada Train / Val / Test

In [4]:
def split_dataset(df, test_size=0.15, val_size=0.15, seed=RANDOM_SEED):
    """
    Estratificado en tres partes. val_size se expresa como fracción del
    dataset original (no del remanente), lo que hace los tamaños predecibles.
    """
    train_val, test = train_test_split(
        df, test_size=test_size, stratify=df[LABEL_COL], random_state=seed
    )
    adjusted_val = val_size / (1.0 - test_size)
    train, val = train_test_split(
        train_val, test_size=adjusted_val,
        stratify=train_val[LABEL_COL], random_state=seed
    )
    logger.info(
        "Split — train: %d, val: %d, test: %d.", len(train), len(val), len(test)
    )
    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True),
    )

train_df, val_df, test_df = split_dataset(df)

# Persistir los splits para que los Notebooks 4 y 5 puedan reconstruirlos
# sin requerir re-ejecución completa de este notebook.
for name, subset in [("train", train_df), ("val", val_df), ("test", test_df)]:
    subset.to_parquet(f"artifacts/{name}_df.parquet", index=False)
logger.info("Splits guardados en artifacts/")

INFO | Split — train: 1737, val: 373, test: 373.
INFO | Splits guardados en artifacts/


## Limpieza de ruido y reducción del vocabulario

El texto crudo de un CV contiene una cantidad considerable de **ruido no semántico**:
signos de puntuación, símbolos especiales, números de teléfono, URLs y caracteres
de formato. Mantener este ruido en el vocabulario genera dos problemas concretos:

- **Dispersión del vocabulario:** cada variante espuria (*Python3*, *Python.*, *python!*)
  se trata como un token distinto, inflando el tamaño del vocabulario y la dimensión
  del espacio de embedding sin añadir información nueva.
- **Ruido en la matriz de co-ocurrencia:** los modelos basados en frecuencias
  (Word2Vec, TF-IDF) y las redes neuronales aprenden representaciones de todos los
  tokens del vocabulario. Incluir tokens no semánticos "contamina" el espacio de
  representación con vectores que no tienen correlato semántico.

### Estrategia de limpieza aplicada

1. **Expresión regular `_NOISE`:** elimina todo carácter que no sea letra latina o
   espacio en blanco. La sustitución se hace por espacio (no por cadena vacía) para
   preservar los límites de palabra: sin esto, `don't` colapsaría a `dont` y
   `C++` desaparecería en lugar de quedar como dos tokens separados.

2. **Eliminación de stopwords (NLTK, inglés):** palabras funcionales de alta frecuencia
   (*the*, *is*, *at*, *which*) no aportan discriminación entre categorías profesionales.
   Su eliminación reduce la dimensión efectiva del vocabulario entre un 20–40 % según
   el corpus, **sin pérdida de semántica profesional**, ya que los términos técnicos
   relevantes no forman parte de la lista de stopwords.

> **Nota importante:** esta función `clean_text()` se define como la **transformación
> canónica única** del pipeline. Se aplicará de forma idéntica durante el
> entrenamiento, la validación y la inferencia en producción, garantizando que no
> exista discrepancia entre la distribución de texto vista por el modelo durante el
> entrenamiento y la que recibe en el momento de la predicción.

## 2.3 Pipeline de limpieza de texto

In [5]:
_NOISE   = re.compile(r"[^a-zA-Z\s]")
_SPACES  = re.compile(r"\s+")
_STOP    = set(stopwords.words("english"))

def clean_text(text: str) -> str:
    """
    Limpieza canónica aplicada de forma idéntica en entrenamiento e inferencia.
    La sustitución por espacio (en lugar de cadena vacía) preserva los límites
    de palabra tras la eliminación de apóstrofes y guiones.
    """
    text = _NOISE.sub(" ", text.lower())
    text = _SPACES.sub(" ", text).strip()
    tokens = [w for w in text.split() if w not in _STOP and len(w) > 1]
    return " ".join(tokens)

# Verificación visual
sample = df[TEXT_COL].iloc[0]
print(f"Original : {sample}")
print(f"Limpio   : {clean_text(sample)}")

Original : ACCOUNTANT
Summary
Financial Accountant specializing in financial planning, reporting and analysis within the Department of Defense.
Highlights
Account reconciliations
Results-oriented
Financial reporting
Critical thinking
Accounting operations professional
Analysis of financial systems
ERP (Enterprise Resource Planning) software.
Excellent facilitator
Accomplishments
Served on a tiger team which identified and resolved General Ledger postings in DEAMS totaling $360B in accounting adjustments. This allowed
for the first successful fiscal year-end close for 2012.
In collaboration with DFAS Europe, developed an automated tool that identified duplicate obligations. This tool allowed HQ USAFE to
deobligate over $5M in duplicate obligations.
Experience
Company Name
 
July 2011
 
to 
November 2012
 
Accountant
 
City
 
, 
State
Enterprise Resource Planning Office (ERO)
In this position as an Accountant assigned to the Defense Enterprise Accounting and Management System (DEAMS) ERO

## 2.4 Vocabulario

In [6]:
class Vocabulary:
    """
    Mapeo bidireccional token ↔ índice entero.
    Índice 0 reservado para PAD (padding_idx=0 en nn.Embedding).
    Índice 1 reservado para UNK para no desplazar índices reales.
    """
    def __init__(self):
        self.token2idx = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
        self.idx2token = {PAD_IDX: PAD_TOKEN, UNK_IDX: UNK_TOKEN}

    def build(self, tokenized_corpus: list, min_freq: int = 2) -> None:
        """
        Se construye únicamente sobre el corpus de entrenamiento.
        min_freq=2 descarta hapax legomena (probablemente errores de ASR).
        """
        freq = Counter(tok for sent in tokenized_corpus for tok in sent)
        for token, count in sorted(freq.items()):
            if count >= min_freq and token not in self.token2idx:
                idx = len(self.token2idx)
                self.token2idx[token] = idx
                self.idx2token[idx]   = token
        logger.info(
            "Vocabulario construido — %d tokens únicos (min_freq=%d).",
            len(self.token2idx), min_freq
        )

    def encode(self, tokens: list) -> list:
        return [self.token2idx.get(t, UNK_IDX) for t in tokens]

    def __len__(self) -> int:
        return len(self.token2idx)

## 2.5 Tokenización y padding adaptativo

In [7]:
def encode_and_pad(
    tokenized_texts: list,
    vocab: Vocabulary,
    seq_len: int,
) -> np.ndarray:
    """
    Trunca conservando los ÚLTIMOS seq_len tokens (tail-truncation): en datasets
    de asistente de voz la intención se concentra al final del enunciado.
    Padding a la derecha para respetar la convención de las LSTM (lectura L→R).
    """
    encoded = np.full((len(tokenized_texts), seq_len), PAD_IDX, dtype=np.int64)
    for i, tokens in enumerate(tokenized_texts):
        indices = vocab.encode(tokens)[-seq_len:]   # tail-truncation
        encoded[i, : len(indices)] = indices
    return encoded


# Tokenizar cada split
train_tokens = [clean_text(t).split() for t in train_df[TEXT_COL]]
val_tokens   = [clean_text(t).split() for t in val_df[TEXT_COL]]
test_tokens  = [clean_text(t).split() for t in test_df[TEXT_COL]]

# Construir vocabulario solo sobre entrenamiento (evita data leakage)
vocab = Vocabulary()
vocab.build(train_tokens, min_freq=2)

# Codificar etiquetas
label_enc = LabelEncoder()
y_train = label_enc.fit_transform(train_df[LABEL_COL])
y_val   = label_enc.transform(val_df[LABEL_COL])
y_test  = label_enc.transform(test_df[LABEL_COL])

# Codificar secuencias
X_train = encode_and_pad(train_tokens, vocab, SEQUENCE_LENGTH)
X_val   = encode_and_pad(val_tokens,   vocab, SEQUENCE_LENGTH)
X_test  = encode_and_pad(test_tokens,  vocab, SEQUENCE_LENGTH)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_val  : {X_val.shape}    y_val  : {y_val.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")
print(f"Clases : {list(label_enc.classes_)}")

INFO | Vocabulario construido — 19311 tokens únicos (min_freq=2).


X_train: (1737, 400)  y_train: (1737,)
X_val  : (373, 400)    y_val  : (373,)
X_test : (373, 400)   y_test : (373,)
Clases : ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']


## Truncamiento, padding y matrices de embeddings estáticos

### Longitud de secuencia fija (`sequence_length`)

Las redes neuronales operan sobre tensores de dimensiones estáticas. Para construir
un batch de secuencias es necesario que todas tengan la misma longitud, lo que
requiere dos operaciones complementarias:

- **Truncamiento** (*tail-truncation*): si un CV supera el límite, se conservan los
  **últimos** `sequence_length` tokens en lugar de los primeros. En documentos de
  currículos, el perfil profesional, las habilidades técnicas y la experiencia reciente
  tienden a concentrarse al final del documento, por lo que la cola del texto es más
  discriminativa que el encabezado.
- **Padding** (*post-padding*): las secuencias más cortas se rellenan con el índice
  reservado `PAD_IDX = 0` al final. El padding a la derecha (y no a la izquierda) es
  consistente con la lectura izquierda-derecha de la BiLSTM: el estado oculto final
  captura contexto real y no tokens vacíos.

El valor concreto de `sequence_length` proviene del **P95 calculado en el Notebook 1**,
garantizando que el 95 % de los documentos no se truncan en absoluto.

### Matrices de embeddings estáticos: Word2Vec y FastText

Un embedding es una función que mapea un token (entero) a un vector denso en
$\mathbb{R}^d$, donde $d$ es la dimensión del espacio de representación. A diferencia
de los embeddings contextuales de un Transformer, estos son **estáticos**: cada token
tiene un único vector independientemente del contexto en que aparece.

| Modelo | Unidad de representación | Ventaja principal |
|--------|--------------------------|-------------------|
| **Word2Vec** (skip-gram) | Palabra completa | Captura relaciones semánticas globales mediante contexto de ventana deslizante |
| **FastText** | Subpalabras (character n-grams, n=3–6) | Robusto ante palabras fuera del vocabulario y errores tipográficos frecuentes en CVs |

Ambas matrices se entrenan **exclusivamente sobre el corpus de entrenamiento** para
evitar data leakage. Su forma resultante es `(vocab_size × embedding_dim)` y serán
cargadas en las capas `nn.Embedding` de la BiLSTM, la CNN-1D y el clasificador
FastText mediante `from_pretrained()`, inicializando los pesos con representaciones
ya significativas en lugar de valores aleatorios.

> **Artefactos exportados a `artifacts/`:** `X_train.npy`, `X_val.npy`, `X_test.npy`,
> `y_*.npy`, `w2v_matrix.npy`, `ft_matrix.npy`, `vocab.pkl`, `label_enc.pkl`,
> `preprocessing_config.json`.

## 2.6 Embeddings — Word2Vec

In [8]:
def _build_embedding_matrix(
    vocab: Vocabulary,
    gensim_model,
    embedding_dim: int,
    model_name: str,
) -> np.ndarray:
    """
    Construye la matriz (vocab_size × embedding_dim) alineada con el vocabulario
    del proyecto. Tokens fuera del vocabulario gensim reciben un vector aleatorio
    N(0, 0.01) en lugar del vector cero: un UNK completamente nulo sería invisible
    para las compuertas de atención y la LSTM.
    """
    rng    = np.random.default_rng(seed=42)
    matrix = rng.normal(0, 0.01, (len(vocab), embedding_dim)).astype(np.float32)
    matrix[PAD_IDX] = 0.0   # el vector de padding debe ser exactamente cero

    covered = 0
    for token, idx in vocab.token2idx.items():
        if token in gensim_model.wv:
            matrix[idx] = gensim_model.wv[token]
            covered += 1

    logger.info(
        "%s — shape=%s, cobertura=%.1f%%.",
        model_name, matrix.shape, covered / len(vocab) * 100
    )
    return matrix


logger.info("Entrenando Word2Vec (skip-gram, dim=%d)...", EMBEDDING_DIM)
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=2,
    sg=1,          # skip-gram: mejor calidad en corpora pequeños
    epochs=10,
    workers=4,
    seed=42,
)
w2v_model.save("models/word2vec.bin")
w2v_matrix = _build_embedding_matrix(vocab, w2v_model, EMBEDDING_DIM, "Word2Vec")
print(f"Matriz Word2Vec: {w2v_matrix.shape}")

INFO | Entrenando Word2Vec (skip-gram, dim=200)...
INFO | collecting all words and their counts
INFO | PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
INFO | collected 32231 word types from a corpus of 1030649 raw words and 1737 sentences
INFO | Creating a fresh vocabulary
INFO | Word2Vec lifecycle event {'msg': 'effective_min_count=2 retains 19309 unique words (59.91% of original 32231, drops 12922)', 'datetime': '2026-05-17T18:38:45.044761', 'gensim': '4.4.0', 'python': '3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-SP0', 'event': 'prepare_vocab'}
INFO | Word2Vec lifecycle event {'msg': 'effective_min_count=2 leaves 1017727 word corpus (98.75% of original 1030649, drops 12922)', 'datetime': '2026-05-17T18:38:45.045861', 'gensim': '4.4.0', 'python': '3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-

Matriz Word2Vec: (19311, 200)


## 2.7 Embeddings — FastText

In [9]:
logger.info("Entrenando FastText (subword n-grams, dim=%d)...", EMBEDDING_DIM)
ft_model = GensimFastText(
    sentences=train_tokens,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=2,
    sg=1,
    epochs=10,
    workers=4,
    seed=42,
    min_n=3,    # rango de character n-grams: maneja errores de ASR
    max_n=6,
)
ft_model.save("models/fasttext.bin")
ft_matrix = _build_embedding_matrix(vocab, ft_model, EMBEDDING_DIM, "FastText")
print(f"Matriz FastText: {ft_matrix.shape}")

INFO | Entrenando FastText (subword n-grams, dim=200)...
INFO | collecting all words and their counts
INFO | PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
INFO | collected 32231 word types from a corpus of 1030649 raw words and 1737 sentences
INFO | Creating a fresh vocabulary
INFO | FastText lifecycle event {'msg': 'effective_min_count=2 retains 19309 unique words (59.91% of original 32231, drops 12922)', 'datetime': '2026-05-17T18:41:29.258165', 'gensim': '4.4.0', 'python': '3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-SP0', 'event': 'prepare_vocab'}
INFO | FastText lifecycle event {'msg': 'effective_min_count=2 leaves 1017727 word corpus (98.75% of original 1030649, drops 12922)', 'datetime': '2026-05-17T18:41:29.259182', 'gensim': '4.4.0', 'python': '3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.

Matriz FastText: (19311, 200)


## 2.8 Exportar artefactos para los Notebooks 3–5

In [10]:
# Arrays de secuencias
np.save("artifacts/X_train.npy", X_train)
np.save("artifacts/X_val.npy",   X_val)
np.save("artifacts/X_test.npy",  X_test)
np.save("artifacts/y_train.npy", y_train)
np.save("artifacts/y_val.npy",   y_val)
np.save("artifacts/y_test.npy",  y_test)

# Matrices de embeddings
np.save("artifacts/w2v_matrix.npy", w2v_matrix)
np.save("artifacts/ft_matrix.npy",  ft_matrix)

# Vocabulario y label encoder
with open("artifacts/vocab.pkl",      "wb") as f: pickle.dump(vocab,     f)
with open("artifacts/label_enc.pkl",  "wb") as f: pickle.dump(label_enc, f)

# Configuración de dimensiones compartida
config = {
    "vocab_size":       len(vocab),
    "num_classes":      len(label_enc.classes_),
    "sequence_length":  SEQUENCE_LENGTH,
    "embedding_dim":    EMBEDDING_DIM,
    "class_names":      list(label_enc.classes_),
}
with open("artifacts/preprocessing_config.json", "w") as f:
    json.dump(config, f, indent=2)

logger.info("Todos los artefactos exportados a artifacts/")
print("\nConfig:")
for k, v in config.items():
    print(f"  {k}: {v}")

INFO | Todos los artefactos exportados a artifacts/



Config:
  vocab_size: 19311
  num_classes: 24
  sequence_length: 400
  embedding_dim: 200
  class_names: ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']
